# TSPTW (Multi-Speed): Route Ending at 160 Broadway

Solves the TSPTW for a configurable range of paces (minutes per mile).
Fixed final stop is 160 Broadway. Configurable service time at each intermediate stop.
All solutions are written to a single database table.

In [1]:
import numpy as np
import pandas as pd
import psycopg
import matplotlib.pyplot as plt

## Parameters

In [2]:
# 0=Monday, 1=Tuesday, 2=Wednesday, 3=Thursday, 4=Friday, 5=Saturday, 6=Sunday
START_DAY = 4  # Saturday

# Start time as HH:MM
START_TIME = "06:00"

# Minutes spent at each intermediate stop (service time); does not apply to final stop
SERVICE_TIME_MIN = 5.0

# Fixed final stop
END_ADDRESS = "160 Broadway"

# Speed range in minutes per mile
PACE_MIN_PER_MI_SLOW = 20.0   # slowest pace to solve
PACE_MIN_PER_MI_FAST =  4.0   # fastest pace to solve
PACE_STEP_MIN_PER_MI =  0.1   # step size (0.1 min = 6 sec per mile)

# --- derived ---
DAYS    = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
DAY_ABBR = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

_h, _m = map(int, START_TIME.split(':'))
START_TIME_MIN = _h * 60 + _m

# Build pace list slow → fast (inclusive on both ends)
PACES = np.arange(
    PACE_MIN_PER_MI_SLOW,
    PACE_MIN_PER_MI_FAST - PACE_STEP_MIN_PER_MI * 0.001,
    -PACE_STEP_MIN_PER_MI
)

print(f"Start:        {DAYS[START_DAY]} at {START_TIME}")
print(f"Service time: {SERVICE_TIME_MIN} min/stop (intermediate stops only)")
print(f"End:          {END_ADDRESS}")
print(f"Paces:        {len(PACES)} values from {PACES[0]:.1f} to {PACES[-1]:.1f} min/mi")

Start:        Friday at 06:00
Service time: 5.0 min/stop (intermediate stops only)
End:          160 Broadway
Paces:        161 values from 20.0 to 4.0 min/mi


## Load data

In [3]:
conn = psycopg.connect("dbname=mctrot host=127.0.0.1 port=5432")

distances = pd.read_sql("""
    SELECT from_mcd_id, to_mcd_id, distance_meters
    FROM route_summary
""", conn)

locations = pd.read_sql("""
    SELECT
        ogc_fid AS mcd_id,
        addressline1,
        restauranthoursmonday,
        restauranthourstuesday,
        restauranthourswednesday,
        restauranthoursthursday,
        restauranthoursfriday,
        restauranthourssaturday,
        restauranthourssunday,
        ST_Y(the_geog::geometry) AS lat,
        ST_X(the_geog::geometry) AS lon
    FROM mcisland_mcd
    ORDER BY ogc_fid
""", conn)

conn.close()
print(f"{len(locations)} locations, {len(distances)} distance pairs")

47 locations, 2162 distance pairs


/var/folders/xc/gjswmb8n415gsl6jjnd178j40000gn/T/ipykernel_37618/3261352032.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  distances = pd.read_sql("""
/var/folders/xc/gjswmb8n415gsl6jjnd178j40000gn/T/ipykernel_37618/3261352032.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  locations = pd.read_sql("""


In [4]:
ids = locations['mcd_id'].tolist()
idx = {mcd_id: i for i, mcd_id in enumerate(ids)}
N   = len(ids)

dist = np.full((N, N), np.inf)
np.fill_diagonal(dist, 0)

for _, row in distances.iterrows():
    i, j = idx[row['from_mcd_id']], idx[row['to_mcd_id']]
    dist[i, j] = row['distance_meters']
    dist[j, i] = row['distance_meters']

# Locate the fixed end stop
end_matches = locations[locations['addressline1'] == END_ADDRESS]
assert len(end_matches) == 1, f"Expected exactly one match for '{END_ADDRESS}', got {len(end_matches)}"
END_POS = locations.index.get_loc(end_matches.index[0])

print(f"Distance matrix: {dist.shape}, missing pairs: {np.isinf(dist).sum()}")
print(f"End stop position in matrix: {END_POS} ({locations.iloc[END_POS]['addressline1']})")

Distance matrix: (47, 47), missing pairs: 0
End stop position in matrix: 46 (160 Broadway)


## Time windows

Parse per-day hours for every location.

Edge cases:
- `00:00 - 00:00` → open all day (0, 1440)
- `HH:MM - 00:00` → closes at midnight, treat close as 1440
- `HH:MM - HH:MM` where close < open → crosses midnight, add 1440 to close

When checking arrivals, also look at the previous day's window in case
it extends past midnight into the current day.

In [5]:
HOUR_COLS = [
    'restauranthoursmonday',
    'restauranthourstuesday',
    'restauranthourswednesday',
    'restauranthoursthursday',
    'restauranthoursfriday',
    'restauranthourssaturday',
    'restauranthourssunday',
]

def parse_hours(hours_str):
    """Parse 'HH:MM - HH:MM' into (open_min, close_min) as minutes from midnight."""
    if not hours_str:
        return (0, 1440)

    open_str, close_str = hours_str.split(' - ')
    oh, om = map(int, open_str.split(':'))
    ch, cm = map(int, close_str.split(':'))

    open_min  = oh * 60 + om
    close_min = ch * 60 + cm

    if open_min == 0 and close_min == 0:
        return (0, 1440)

    if close_min == 0:
        close_min = 1440
    elif close_min < open_min:
        close_min += 1440

    return (open_min, close_min)


windows = [
    {d: parse_hours(row[col]) for d, col in enumerate(HOUR_COLS)}
    for _, row in locations.iterrows()
]

print("Locations with restricted hours (any day):")
for i, (_, row) in enumerate(locations.iterrows()):
    restricted = [
        f"{DAY_ABBR[d]}: {o//60:02d}:{o%60:02d}-{c//60%24:02d}:{c%60:02d}"
        for d, (o, c) in windows[i].items()
        if o > 0 or c < 1440
    ]
    if restricted:
        print(f"  {row['addressline1']}")
        for r in restricted:
            print(f"    {r}")

Locations with restricted hours (any day):
  2379 Adam Clayton Powell Jr Blvd
    Mon: 06:00-00:00
    Tue: 06:00-00:00
    Wed: 06:00-00:00
    Thu: 06:00-00:00
    Fri: 06:00-00:00
    Sat: 06:00-00:00
    Sun: 06:00-00:00
  354 W 125th St
    Mon: 06:00-23:00
    Tue: 06:00-23:00
    Wed: 06:00-23:00
    Thu: 06:00-00:00
    Fri: 06:00-00:00
    Sat: 06:00-00:00
    Sun: 06:00-23:00
  600a W. 125 Street Broadway
    Mon: 06:00-23:00
    Tue: 06:00-23:00
    Wed: 06:00-23:00
    Thu: 06:00-00:00
    Fri: 06:00-00:00
    Sat: 06:00-00:00
    Sun: 06:00-23:00
  3809 Broadway
    Mon: 06:00-23:00
    Tue: 06:00-23:00
    Wed: 06:00-23:00
    Thu: 06:00-02:00
    Fri: 06:00-02:00
    Sat: 06:00-02:00
    Sun: 06:00-23:00
  1872 3rd Ave
    Mon: 06:00-23:00
    Tue: 06:00-23:00
    Wed: 06:00-23:00
    Thu: 06:00-23:00
    Fri: 06:00-00:00
    Sat: 06:00-00:00
    Sun: 06:00-23:00
  1871 2nd Ave
    Mon: 06:00-23:00
    Tue: 06:00-23:00
    Wed: 06:00-23:00
    Thu: 06:00-23:00
    Fri: 0

## TSPTW solver (fixed end, multi-speed)

160 Broadway is pinned to the last position. 2-opt moves are restricted to
positions `0..N-2` so it cannot be displaced.

Service time is added after each intermediate stop (not the final stop).

In [6]:
VIOLATION_PENALTY = 30 * 24 * 60  # 30 days in minutes


def open_at(windows, loc_idx, start_day, absolute_time):
    """Return (is_open, minutes_to_wait) for loc_idx at the given absolute time."""
    day_offset  = int(absolute_time // 1440)
    time_in_day = absolute_time % 1440
    dow         = (start_day + day_offset) % 7
    open_min, close_min = windows[loc_idx][dow]

    if open_min <= time_in_day < close_min:
        return True, 0.0

    if day_offset > 0:
        prev_dow = (start_day + day_offset - 1) % 7
        prev_open, prev_close = windows[loc_idx][prev_dow]
        if prev_close > 1440 and time_in_day < prev_close - 1440:
            return True, 0.0

    if time_in_day < open_min:
        return False, float(open_min - time_in_day)

    next_dow = (dow + 1) % 7
    next_open, _ = windows[loc_idx][next_dow]
    wait = (1440 - time_in_day) + next_open
    return False, float(wait)


def route_cost(route, dist, windows, start_day, start_time_min, speed_m_per_min, service_time_min, end_pos):
    """Total route cost in minutes (travel + waiting + service) plus violation penalties.

    Service time is applied at every stop except the fixed final stop (end_pos).
    """
    current_time = float(start_time_min)
    violations   = 0
    last_step    = len(route) - 1

    for step, loc_idx in enumerate(route):
        if step > 0:
            travel  = dist[route[step - 1], loc_idx] / speed_m_per_min
            arrival = current_time + travel
        else:
            arrival = current_time

        is_open, wait = open_at(windows, loc_idx, start_day, arrival)
        service = 0.0 if step == last_step else service_time_min

        if is_open:
            current_time = arrival + service
        elif wait > 0 and wait < 1440:
            current_time = arrival + wait + service
        else:
            violations  += 1
            current_time = arrival + service

    return (current_time - start_time_min) + violations * VIOLATION_PENALTY

In [7]:
def simulated_annealing_tsptw(
    dist, windows, start_day, start_time_min, speed_m_per_min, end_pos,
    service_time_min=0.0,
    T_start=5000, T_end=0.1, cooling=0.9995, seed=42
):
    rng = np.random.default_rng(seed)
    N   = len(dist)

    others = [i for i in range(N) if i != end_pos]
    route  = list(rng.permutation(others)) + [end_pos]

    def cost(r):
        return route_cost(r, dist, windows, start_day, start_time_min,
                          speed_m_per_min, service_time_min, end_pos)

    best_route = route[:]
    best_cost  = cost(route)
    cur_cost   = best_cost
    T          = T_start
    iters      = 0
    M          = N - 1  # free positions (end_pos is fixed at route[-1])

    while T > T_end:
        i, j = sorted(rng.choice(M, 2, replace=False))
        candidate      = route[:]
        candidate[i:j+1] = reversed(candidate[i:j+1])

        c_new = cost(candidate)
        delta = c_new - cur_cost

        if delta < 0 or rng.random() < np.exp(-delta / T):
            route    = candidate
            cur_cost = c_new
            if c_new < best_cost:
                best_cost  = c_new
                best_route = route[:]

        T     *= cooling
        iters += 1

    return best_route, best_cost, iters

## Solve for all paces

In [8]:
# Each entry: (pace_min_per_mi, route, cost, violations, net_min)
results = []

for pace in PACES:
    speed_m_per_min = 1609.344 / pace
    route, cost, iters = simulated_annealing_tsptw(
        dist, windows, START_DAY, START_TIME_MIN, speed_m_per_min, END_POS,
        service_time_min=SERVICE_TIME_MIN
    )
    violations = int(cost // VIOLATION_PENALTY)
    net_min    = cost - violations * VIOLATION_PENALTY
    results.append((pace, route, cost, violations, net_min))
    flag = " *** VIOLATION" if violations else ""
    print(f"pace={pace:5.1f} min/mi  violations={violations}  time={net_min:6.0f} min ({net_min/60:.1f} h){flag}")

print(f"\nDone. Solved {len(results)} paces.")

pace= 20.0 min/mi  violations=0  time=  1098 min (18.3 h)
pace= 19.9 min/mi  violations=0  time=  1456 min (24.3 h)
pace= 19.8 min/mi  violations=0  time=  1428 min (23.8 h)
pace= 19.7 min/mi  violations=0  time=   810 min (13.5 h)
pace= 19.6 min/mi  violations=0  time=  1491 min (24.9 h)
pace= 19.5 min/mi  violations=0  time=   835 min (13.9 h)
pace= 19.4 min/mi  violations=0  time=  1491 min (24.8 h)
pace= 19.3 min/mi  violations=0  time=   786 min (13.1 h)
pace= 19.2 min/mi  violations=0  time=  1491 min (24.8 h)
pace= 19.1 min/mi  violations=0  time=  1491 min (24.8 h)
pace= 19.0 min/mi  violations=0  time=  1455 min (24.3 h)
pace= 18.9 min/mi  violations=0  time=   792 min (13.2 h)
pace= 18.8 min/mi  violations=0  time=  1455 min (24.3 h)
pace= 18.7 min/mi  violations=0  time=  1540 min (25.7 h)
pace= 18.6 min/mi  violations=0  time=  1455 min (24.3 h)
pace= 18.5 min/mi  violations=0  time=   794 min (13.2 h)
pace= 18.4 min/mi  violations=0  time=  1455 min (24.2 h)
pace= 18.3 min

## Summary

In [9]:
summary = pd.DataFrame([
    {
        'pace_min_per_mi': pace,
        'violations':      violations,
        'total_min':       round(net_min, 1),
        'total_hr':        round(net_min / 60, 2),
    }
    for pace, route, cost, violations, net_min in results
])

def highlight_violations(row):
    if row['violations'] > 0:
        return ['background-color: #ffcccc'] * len(row)
    return [''] * len(row)

summary.style.apply(highlight_violations, axis=1)

,pace_min_per_mi,violations,total_min,total_hr
0,20.000000,0,1098.400000,18.310000
1,19.900000,0,1455.700000,24.260000
2,19.800000,0,1427.700000,23.800000
3,19.700000,0,810.200000,13.500000
4,19.600000,0,1491.100000,24.850000
5,19.500000,0,835.100000,13.920000
6,19.400000,0,1490.900000,24.850000
7,19.300000,0,785.600000,13.090000
8,19.200000,0,1490.700000,24.850000
9,19.100000,0,1490.700000,24.840000


## Detailed route analysis

`analyze_route` reconstructs the per-stop timeline for a given solution.
Call it with any `(pace, route)` from `results` to inspect that solution.

In [10]:
def fmt_time(absolute_min):
    day_offset  = int(absolute_min // 1440)
    time_in_day = absolute_min % 1440
    dow = (START_DAY + day_offset) % 7
    h   = int(time_in_day // 60)
    m   = int(time_in_day % 60)
    return f"{DAY_ABBR[dow]} {h:02d}:{m:02d}"


def analyze_route(route, pace_min_per_mi, service_time_min):
    """Build a per-stop DataFrame for the given route and pace."""
    speed_m_per_min = 1609.344 / pace_min_per_mi
    current_time    = float(START_TIME_MIN)
    last_step       = len(route) - 1
    rows            = []

    for step, loc_idx in enumerate(route):
        if step > 0:
            travel  = dist[route[step - 1], loc_idx] / speed_m_per_min
            arrival = current_time + travel
        else:
            arrival = current_time
            travel  = 0.0

        is_open, wait = open_at(windows, loc_idx, START_DAY, arrival)
        service = 0.0 if step == last_step else service_time_min

        day_offset  = int(arrival // 1440)
        dow         = (START_DAY + day_offset) % 7
        time_in_day = arrival % 1440
        open_min, close_min = windows[loc_idx][dow]

        if is_open:
            status       = 'ok'
            current_time = arrival + service
        elif wait > 0:
            status       = 'wait'
            current_time = arrival + wait + service
        else:
            status       = 'VIOLATION'
            current_time = arrival + service

        rows.append({
            'stop':             step + 1,
            'address':          locations.iloc[loc_idx]['addressline1'],
            'travel_min':       round(travel, 1),
            'arrival':          fmt_time(arrival),
            'time_window':      f"{open_min//60:02d}:{open_min%60:02d}\u2013"
                                f"{(close_min//60)%24:02d}:{close_min%60:02d}"
                                + ("+1" if close_min > 1440 else ""),
            'minutes_to_close': round(close_min - time_in_day, 1),
            'wait_min':         round(wait, 1) if status == 'wait' else 0,
            'service_min':      service,
            'status':           status,
        })

    return pd.DataFrame(rows)

In [11]:
# Display detailed analysis for a specific pace — change index as desired
INSPECT_PACE = 10.0  # min/mi

match = [(p, r) for p, r, *_ in results if abs(p - INSPECT_PACE) < 0.001]
if not match:
    print(f"Pace {INSPECT_PACE} not in results. Available: {[p for p,*_ in results]}")
else:
    _pace, _route = match[0]
    _analysis = analyze_route(_route, _pace, SERVICE_TIME_MIN)

    def highlight(row):
        if row['status'] == 'VIOLATION':
            return ['background-color: #ffcccc'] * len(row)
        if row['status'] == 'wait':
            return ['background-color: #fff3cc'] * len(row)
        return [''] * len(row)

    print(f"Pace: {_pace} min/mi")
    display(_analysis.style.apply(highlight, axis=1))

Pace: 9.999999999999858 min/mi


,stop,address,travel_min,arrival,time_window,minutes_to_close,wait_min,service_min,status
0,1,966 3rd Ave,0.000000,Fri 06:00,00:00–00:00,1080.000000,0,5.000000,ok
1,2,1286 1st Ave,8.000000,Fri 06:13,00:00–00:00,1067.000000,0,5.000000,ok
2,3,1499 3rd Ave,10.900000,Fri 06:28,00:00–00:00,1051.100000,0,5.000000,ok
3,4,1871 2nd Ave,6.900000,Fri 06:40,06:00–23:00,979.200000,0,5.000000,ok
4,5,1872 3rd Ave,4.300000,Fri 06:50,06:00–00:00,1029.900000,0,5.000000,ok
5,6,2142 3rd Ave,7.100000,Fri 07:02,00:00–00:00,1017.800000,0,5.000000,ok
6,7,354 W 125th St,12.500000,Fri 07:19,06:00–00:00,1000.300000,0,5.000000,ok
7,8,148 W 125th Street,2.600000,Fri 07:27,00:00–00:00,992.700000,0,5.000000,ok
8,9,2379 Adam Clayton Powell Jr Blvd,7.000000,Fri 07:39,06:00–00:00,980.700000,0,5.000000,ok
9,10,608 West 207th Street,37.700000,Fri 08:22,00:00–00:00,938.000000,0,5.000000,ok


## Save to database

In [12]:
conn = psycopg.connect("dbname=mctrot host=127.0.0.1 port=5432")

with conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS tsptw_multi_route")
    cur.execute("""
        CREATE TABLE tsptw_multi_route (
            pace_min_per_mi   NUMERIC,
            stop_order        INTEGER,
            mcd_id            INTEGER,
            addressline1      TEXT,
            arrival           TEXT,
            time_window       TEXT,
            minutes_to_close  NUMERIC,
            travel_min        NUMERIC,
            wait_min          NUMERIC,
            service_min       NUMERIC,
            status            TEXT,
            PRIMARY KEY (pace_min_per_mi, stop_order)
        )
    """)

    total_rows = 0
    for pace, route, cost, violations, net_min in results:
        analysis = analyze_route(route, pace, SERVICE_TIME_MIN)
        for _, row in analysis.iterrows():
            loc_idx = route[int(row['stop']) - 1]
            cur.execute(
                """
                INSERT INTO tsptw_multi_route
                    (pace_min_per_mi, stop_order, mcd_id, addressline1,
                     arrival, time_window, minutes_to_close,
                     travel_min, wait_min, service_min, status)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                """,
                (
                    float(pace),
                    int(row['stop']),
                    int(ids[loc_idx]),
                    row['address'],
                    row['arrival'],
                    row['time_window'],
                    row['minutes_to_close'],
                    row['travel_min'],
                    row['wait_min'],
                    row['service_min'],
                    row['status'],
                )
            )
        total_rows += len(analysis)

conn.commit()
conn.close()
print(f"Written {total_rows} rows ({len(results)} paces × {len(results[0][1])} stops) to tsptw_multi_route.")

Written 7567 rows (161 paces × 47 stops) to tsptw_multi_route.
